# Bulk upgrade your Fabric workspaces to Runtime 2.0

New Fabric workspaces use Runtime 2.0 by default, but existing workspaces stay on the runtime they were created with. Clicking through the workspace settings is fine for one or two workspaces. For a few dozen workspaces, this notebook does it in one go.

**What it does**
1. Lists all workspaces you have access to.
2. Filters them on a name pattern (for example `DEV`).
3. Sets the default Spark runtime of each matching workspace to `2.0` using [semantic-link-labs](https://github.com/microsoft/semantic-link-labs).

**Before you run it**
- Run this in a **Python notebook** in Fabric. It only calls the Fabric REST API, so it doesn't need a Spark cluster.
- You need the **Admin** role on every workspace you want to update.
- A runtime upgrade is also a Spark, Python and library upgrade. Test your notebooks and pipelines on Runtime 2.0 in one workspace before you roll it out to all of them.
- This changes the **workspace default** runtime. Notebooks attached to an environment use the runtime of that environment, so update those environments separately.

**Author:** Joppe Wouters  
[**More Fabric tips & tricks**](https://github.com/JoppeWouters/fabric-tips-and-tricks)

## 1. Install semantic-link-labs

In [ ]:
%pip install semantic-link-labs -q

## 2. Parameters

In [ ]:
import sempy_labs
import sempy.fabric

workspace_filter = "DEV"
dry_run = True

## 3. Find the matching workspaces

Check this list carefully before you switch off the dry run. Every workspace in it will be updated.

In [ ]:
df_workspaces = sempy.fabric.list_workspaces()["Name"]

df_workspaces_filter = df_workspaces[df_workspaces.str.contains(workspace_filter, case=False, na=False)]
display(df_workspaces_filter)

## 4. Upgrade to Runtime 2.0

Loops over the filtered workspaces and calls `sempy_labs.spark.update_spark_settings()` with `runtime_version="2.0"`. Only the runtime changes. All other Spark settings stay as they are.

In [ ]:
if not dry_run:
    for workspace_name in df_workspaces_filter:
        sempy_labs.spark.update_spark_settings(
            workspace=workspace_name,
            runtime_version="2.0"
        )
else:
    print("Dry run enabled. No changes will be made.")